# Simulation study: fixed-domain MLE for the RBF kernel with a nugget

This notebook accompanies *Sharp Asymptotic Theory of Maximum Likelihood Estimation for Gaussian Processes with an RBF Kernel* (A. Qaqish and D. Li). All computation is in `rbf_mle_sim.py`; the notebook calls it, checks it against brute-force linear algebra, draws the figures, and computes every number quoted in Section 4 and Appendix D of the paper (Section 8 below).

**Model.** $Y_i = F(x_i) + \varepsilon_i$ on $[0,1]^p$, $p\in\{1,2,3\}$, where $F\sim\mathrm{GP}\big(0,\ \sigma^2 e^{-\|x-x'\|^2/(2\ell^2)}\big)$ and $\varepsilon_i\sim N(0,\tau^2)$. The design is a regular midpoint grid with $G$ points per axis, $n=G^p$. True values: $\sigma_0^2=1,\ \ell_0=0.25,\ \tau_0^2=0.01$.

**What the theorem predicts** (with $b_n=\log n/\log\log n$):

| parameter | rate | log–log check |
|---|---|---|
| $\sigma^2$ | $b_n^{-p/2}$ | RMSE vs $b_n$, slope $-p/2$ |
| $\ell$ | $b_n^{-(p+2)/2}$ | RMSE vs $b_n$, slope $-(p+2)/2$ |
| $\tau^2$ | $n^{-1/2}$, with $\sqrt n(\hat\tau^2-\tau_0^2)\Rightarrow N(0,2\tau_0^4)$ | RMSE vs $n$, slope $-1/2$, constant $\sqrt2\tau_0^2$ |

Part (v) of the theorem also gives $\mathbb E(\hat\theta_j-\theta_{0j})^2 \approx [\mathcal I_n(\theta_0)^{-1}]_{jj}$, and part (iii) gives $\mathcal I_n^{1/2}(\hat\theta-\theta_0)\Rightarrow N_3(0,I_3)$.

**How the study is designed.**

1. *Likelihood at large $n$.* The power series in the proof, $e^{-(x-x')^2/2\ell^2}=\sum_k\phi_k(x)\phi_k(x')$ with $\phi_k(x)=e^{-u^2/2}u^k/\sqrt{k!}$ and $u=(x-\tfrac12)/\ell$, reproduces the kernel to about $10^{-14}$ with $M\approx 50$–$225$ terms. So $R_n=\Phi\Phi^\top$ has numerical rank $M\ll n$, and the Gaussian likelihood, exact draws from the model and the Fisher information cost $O(nM^2)$ instead of $O(n^3)$. For $p\ge2$ the grid kernel is a Kronecker product $R_G\otimes\cdots\otimes R_G$. This makes the global MLE feasible up to $n=10^6$ in every dimension. Section 1 checks all of this against dense $O(n^3)$ linear algebra.
2. *Global MLE.* The profile likelihood in $\ell$ can have more than one local maximum, so it is evaluated on a 25-point log grid over $\Theta_\ell=[0.05,2]$, maximising over $(\sigma^2,\tau^2)\in[10^{-2},10^2]\times[10^{-4},1]$ at each $\ell$ (L-BFGS-B from several starts). Every local maximum on the grid is then refined by a bounded Brent search in $\log\ell$, and the best point is the MLE.
3. *How to read logarithmic rates.* Over any feasible range, $b_n$ changes by only a factor of about 2, so a slope fitted to RMSE alone is a weak test. We therefore also compute the exact **Fisher-information curve** $[\mathcal I_n^{-1}]_{jj}^{1/2}$ out to $n=10^8$ ($p=1$) and $n=10^{12}$ ($p=2,3$). By the theorem, this curve is the asymptotic RMSE. Two things should then hold: (a) the Monte Carlo RMSE tracks the Fisher curve, and (b) the Fisher curve becomes parallel to the $b_n^{-p/2}$ and $b_n^{-(p+2)/2}$ reference lines.
4. *Replicates and normality.* 1000 replicates for each $(p,n)$. Bootstrap 95% intervals are shown for the RMSE, normal Q–Q plots for the standardized errors, and a joint check that $\|\mathcal I_n^{1/2}(\hat\theta-\theta_0)\|^2\approx\chi^2_3$.

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
from joblib import Parallel, delayed
import rbf_mle_sim as sim

print("theta0 =", sim.THETA0, "\nTheta  =", sim.BOUNDS)
for p in (1, 2, 3):
    print(f"n (p={p}):", sim.N_LIST[p])
print("replicates:", sim.N_REP)

## 1. Correctness checks against dense $O(n^3)$ code

The fast likelihood, the global MLE and the Fisher information should agree with brute-force dense linear algebra on problems small enough for it (Appendix D.9 of the paper): $n=300$, $1000$ and $4000$ for $p=1$, $G=20$ for $p=2$ and $G=6$ for $p=3$. The log-likelihood is compared at $\theta_0$ and three other parameter values, and the Fisher information with $\tfrac12\operatorname{tr}(C^{-1}C_aC^{-1}C_b)$ computed densely. The MLE is compared with a Nelder–Mead search of the dense likelihood for $n\le1000$; beyond that the search takes too long. A second check confirms that the simulated data have covariance $C_n(\theta_0)$. This cell takes a few minutes.

In [ ]:
def dense_design(p, G):
    # the G^p midpoint grid as an n x p array, in the same order as the data arrays
    g = sim.grid(G)
    return np.stack(np.meshgrid(*([g] * p), indexing="ij"), -1).reshape(-1, p)

rng = np.random.default_rng(1)
thetas = [(1.0, .25, .01), (2.3, .4, .05), (.4, .12, .002), (5, .06, .3)]
for p, G in [(1, 300), (1, 1000), (1, 4000), (2, 20), (3, 6)]:
    t0 = time.time()
    n = G ** p
    Y = sim.simulate(p, n, sim.THETA0, rng)
    X, y, E = dense_design(p, G), Y.ravel(), float((Y ** 2).sum())
    err = []
    for th in thetas:
        lam, z2 = sim._suffstats(p, Y, th[1])
        f, _ = sim._nll_st(np.log(th[0]), np.log(th[2]), lam, z2, n, E)
        d = sim.nll_dense(X, y, th)
        err.append([abs(f - d), abs(f - d) / abs(d)])
    err = np.max(err, axis=0)
    print(f"p={p}, n={n}: max |loglik error| = {err[0]:.1e} (relative {err[1]:.1e})")
    if n <= 1000:
        est = sim.fit_mle(p, Y)
        ref = minimize(lambda v: sim.nll_dense(X, y, np.exp(v)), np.log(est) + 0.02, method="Nelder-Mead",
                       options=dict(xatol=1e-8, fatol=1e-10, maxiter=4000))
        print(f"   MLE (fast)  = {np.round(est, 6)}\n   MLE (dense) = {np.round(np.exp(ref.x), 6)}")
    I1, I2 = sim.fisher_information(p, n, sim.THETA0), sim.fisher_dense(X, sim.THETA0)
    print(f"   Fisher information relative error = {np.abs(I1 - I2).max() / np.abs(I2).max():.1e}   [{time.time() - t0:.0f}s]")

# data generation: covariance of 2 x 10^4 simulated data sets vs C_n(theta0)
for p, G in [(1, 40), (2, 6), (3, 4)]:
    X = dense_design(p, G); n = len(X)
    D2 = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)
    C = sim.THETA0["s"] * np.exp(-D2 / (2 * sim.THETA0["l"] ** 2)) + sim.THETA0["t"] * np.eye(n)
    Ys = np.array([sim.simulate(p, n, sim.THETA0, rng).ravel() for _ in range(20000)])
    Ce = Ys.T @ Ys / len(Ys)
    se = np.sqrt((np.outer(np.diag(C), np.diag(C)) + C ** 2) / len(Ys))   # SD of each entry of Ce
    z = (np.abs(Ce - C) / se)[np.triu_indices(n)]                       # standardized deviations
    print(f"p={p}, n={n}: {z.size} covariance entries; mean squared standardized deviation {np.mean(z ** 2):.2f} "
          f"(expected 1), {np.mean(z > 1.96):.1%} beyond 1.96 (expected 5%), largest {z.max():.2f}")

## 2. Run (or load) the simulation

The full study (`python rbf_mle_sim.py`) takes several hours on a laptop; most of that is $p=1$ with $n\ge3\times10^5$. The saved results in `results/` are loaded when present. Set `RUN = "quick"` for a 3-minute version with fewer sample sizes and 200 replicates (written to `results_quick/` and `figures_quick/`), or `RUN = "full"` to redo everything (this overwrites `results/` and `figures/`).

In [ ]:
RUN = "load"   # "load" | "quick" | "full"
DIMS = (1, 2, 3)  # dimensions to (re)run when RUN != "load"
RESDIR = "results_quick" if RUN == "quick" else "results"
FIGDIR = "figures_quick" if RUN == "quick" else "figures"

if RUN in ("quick", "full"):
    cfg = sim.QUICK if RUN == "quick" else dict(N_LIST=sim.N_LIST, N_REP=sim.N_REP, N_FISHER=sim.N_FISHER)
    for p in DIMS:
        sim.run_all(p, cfg, outdir=RESDIR)

PS = [p for p in (1, 2, 3) if os.path.exists(os.path.join(RESDIR, f"sim_p{p}.npz"))]
res = {p: sim.load(os.path.join(RESDIR, f"sim_p{p}.npz")) for p in PS}
for p in PS:
    sim.print_table(res[p])

The table shows, on the original scale: Monte Carlo RMSE next to the asymptotic SD $[\mathcal I_n(\theta_0)^{-1}]_{jj}^{1/2}$ (columns 'asy.SD'); $\sqrt n\,\mathrm{RMSE}(\hat\tau^2)$ next to its limit $\sqrt2\,\tau_0^2$; and the share of estimates lying on the boundary of $\Theta$.

## 3. Figures 1–3 of the paper (log scale)

The paper reports results for $(\log\sigma^2,\log\ell,\log\tau^2)$. By the delta method the rates are unchanged, the asymptotic SD becomes $[\mathcal I_n(\theta_0)^{-1}]_{jj}^{1/2}/\theta_{0j}$, and $\sqrt n(\log\hat\tau^2-\log\tau_0^2)\Rightarrow N(0,2)$ with no unknown constant. The normal approximation is also visibly better on this scale.

**Top row:** log–log rate plots. For $\sigma^2$ and $\ell$ the horizontal axis is $b_n$ (with $n$ on the top axis), so the theoretical rate is a straight line. Filled circles are Monte Carlo RMSEs, which check the moment statement (v). Open squares are the robust spread $\mathrm{IQR}/1.349$, which checks the $O_{\mathbb P}$ rates (ii) and is not driven by a few large estimates. Both carry bootstrap 95% intervals. The grey line is the asymptotic SD. It is not fitted to the simulations: for each $n$ it is computed exactly from the expected Fisher information $[\mathcal I_n(\theta_0)]_{ab}=\tfrac12\operatorname{tr}(C^{-1}\partial_aC\,C^{-1}\partial_bC)$ of the actual grid design at the true parameter. It needs no simulation, so it is extended into the shaded region beyond the simulated range. The dashed line is the theoretical rate, anchored at the right end; for $\tau^2$ it has no free constant.

**Bottom row:** normal Q–Q plots of the standardized errors at small, medium and the largest $n$, shaded from light to dark as $n$ grows. The grey band is the pointwise 95% range for 1000 draws from an exact $N(0,1)$. Points beyond the frame are drawn as triangles on its top or bottom edge.

The figures are saved as `figures/rbf_mle_log_p{1,2,3}.pdf` (and `.png`).

In [ ]:
os.makedirs(FIGDIR, exist_ok=True)
for p in PS:
    sim.make_figure(res[p], os.path.join(FIGDIR, f"rbf_mle_log_p{p}.pdf"), log=True)
    plt.show()

### Figures A1–A3 (original scale)

The same figures for $(\sigma^2,\ell,\tau^2)$ themselves, saved as `figures/rbf_mle_p{1,2,3}.pdf`. On this scale $\hat\sigma^2=\exp(\log\hat\sigma^2)$ inherits a lognormal-type right tail when only a few Taylor coefficients are effectively available (as for $p=1$). Section 8 checks that such large values are global maxima of the likelihood on the $\sigma^2$–$\ell$ ridge, not optimisation failures.

In [ ]:
for p in PS:
    sim.make_figure(res[p], os.path.join(FIGDIR, f"rbf_mle_p{p}.pdf"))
    plt.show()

## 4. Bounded rescaled error

The theorem states rates up to constants: $b_n^{p/2}\,\mathrm{RMSE}(\hat\sigma^2)$, $b_n^{(p+2)/2}\,\mathrm{RMSE}(\hat\ell)$ and $\sqrt n\,\mathrm{RMSE}(\hat\tau^2)$ should stay bounded away from $0$ and $\infty$. For comparison, the $\sigma^2$ panel also shows $\sqrt n\,\mathrm{RMSE}$, which a parametric $n^{-1/2}$ rate would require to be bounded. It diverges, confirming that $\sigma^2$ and $\ell$ are only logarithmically estimable.

In [ ]:
fig, axes = plt.subplots(len(PS), 3, figsize=(12, 3.25 * len(PS)), squeeze=False)
for r, p in enumerate(PS):
    S = sim.summarize(res[p]); n, nF = S["n"], res[p]["n_fisher"]
    sdF = np.sqrt(np.diagonal(res[p]["V_fisher"], axis1=1, axis2=2))
    scal = [lambda m: sim.b_n(m) ** (p / 2), lambda m: sim.b_n(m) ** ((p + 2) / 2), lambda m: np.sqrt(m)]
    names = [r"$b_n^{p/2}$", r"$b_n^{(p+2)/2}$", r"$\sqrt{n}$"]
    for j in range(3):
        ax = axes[r, j]
        ax.plot(nF, scal[j](nF) * sdF[:, j], "-", color="0.55", label="asymptotic SD")
        ax.errorbar(n, scal[j](n) * S["rmse"][:, j],
                    yerr=[scal[j](n) * (S["rmse"][:, j] - S["lo"][:, j]), scal[j](n) * (S["hi"][:, j] - S["rmse"][:, j])],
                    fmt="o", ms=4, color=sim.COL[j], capsize=2, label="MLE")
        if j == 2: ax.axhline(np.sqrt(2) * res[p]["theta0"][2], ls="--", color="k", label=r"$\sqrt{2}\tau_0^2$")
        if j == 0:
            ax2 = ax.twinx(); ax2.plot(n, np.sqrt(n) * S["rmse"][:, 0], "x:", color="0.3", label=r"$\sqrt{n}\,$RMSE")
            ax2.set_yscale("log"); ax2.set_ylabel(r"$\sqrt{n}\,$RMSE (diverges)", color="0.3"); ax2.legend(loc="lower right", frameon=False, fontsize=8)
        ax.set_xscale("log"); ax.set_ylim(0, None); ax.set_xlabel("$n$")
        ax.set_title(f"p={p}: {names[j]} × RMSE of {sim.PAR_LABELS[j]}", fontsize=10)
        ax.legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout(); plt.show()

## 5. Local slopes of the asymptotic SD (Appendix D.8)

Local slopes $d\log[\mathcal I_n^{-1}]_{jj}^{1/2}/d\log b_n$ show how fast the asymptotic exponents are reached. They are the same on the original and the log scale. Because the rates are doubly logarithmic, pre-asymptotic effects (for example, the domain-to-lengthscale ratio, which fixes how many Taylor coefficients are "free" at small $n$) fade only slowly. The first three lines for each $p$ are the slopes quoted in the paper: between the smallest simulated $n$ and $n\approx10^3$, between the two largest simulated $n$ ("near $n=10^6$"), and over the last interval of the Fisher curve.

In [ ]:
def local_slope(nn, sd, a, b):
    # slope of log sd against log b_n between nn[a] and nn[b], for sigma^2 and ell
    return (np.log(sd[b, :2]) - np.log(sd[a, :2])) / (np.log(sim.b_n(nn[b])) - np.log(sim.b_n(nn[a])))

for p in PS:
    n = res[p]["n"]; sd = np.sqrt(np.diagonal(res[p]["V"], axis1=1, axis2=2))
    nF = res[p]["n_fisher"]; sdF = np.sqrt(np.diagonal(res[p]["V_fisher"], axis1=1, axis2=2))
    k3 = int(np.argmin(np.abs(np.log(n / 1e3))))           # simulated n closest to 10^3
    print(f"p={p}: target slopes {-p/2:g} (sigma^2), {-(p+2)/2:g} (ell)")
    for lab, nn, s, a, b in [("", n, sd, 0, k3), ("", n, sd, -2, -1), ("  (end of the Fisher curve)", nF, sdF, -2, -1)]:
        sl = local_slope(nn, s, a, b)
        print(f"   n = {nn[a]:.3g} to {nn[b]:.3g}:  slope sigma^2 = {sl[0]:6.3f}   slope ell = {sl[1]:6.3f}{lab}")
    print("   all intervals of the Fisher curve:")
    for k in range(len(nF) - 1):
        sl = local_slope(nF, sdF, k, k + 1)
        print(f"      n = {nF[k]:9.3g} to {nF[k + 1]:9.3g}:  {sl[0]:6.3f}  {sl[1]:6.3f}")

## 6. Normality: summary statistics and the joint $\chi^2_3$ check (Appendix D.6)

For each $n$, the tables list the skewness and excess kurtosis of each standardized coordinate (both $0$ under normality), the Kolmogorov–Smirnov distance to $N(0,1)$, and the mean of $Q=\|\mathcal I_n^{1/2}(\hat\theta-\theta_0)\|^2$, which is $3$ under the joint normal limit. The paper reports the log scale; the original scale is shown for comparison. On the original scale $\hat\sigma^2$ is right-skewed: it effectively averages only about $b_n^p$ squared Taylor coefficients, like a scaled $\chi^2$ with few degrees of freedom. The skewness decreases with $n$, more quickly for larger $p$. The figure is the joint Q–Q plot of $Q$ at the largest $n$, on the log scale.

In [ ]:
for scale in ("log", "original"):
    print(f"\n=== {scale} scale ===")
    for p in PS:
        r = sim.to_log(res[p]) if scale == "log" else res[p]
        print(f"p = {p}:      skewness (s,l,t)       excess kurtosis (s,l,t)   KS distance (s,l,t)    mean Q (target 3)")
        for i, n in enumerate(r["n"]):
            z, q = sim.standardized(r, i)
            sk, ku = stats.skew(z), stats.kurtosis(z)
            ks = [stats.kstest(z[:, j], "norm").statistic for j in range(3)]
            print(f"  n={n:>8d}  " + " ".join(f"{v:6.2f}" for v in sk) + "   " + " ".join(f"{v:6.2f}" for v in ku)
                  + "   " + " ".join(f"{v:5.3f}" for v in ks) + f"   {q.mean():6.2f}")

fig, axes = plt.subplots(1, len(PS), figsize=(4.5 * len(PS), 3.6), squeeze=False)
for ax, p in zip(axes[0], PS):
    r = sim.to_log(res[p]); i = len(r["n"]) - 1; _, q = sim.standardized(r, i)
    qs = np.sort(q); th = stats.chi2.ppf((np.arange(1, q.size + 1) - 0.5) / q.size, 3)
    ax.plot(th, qs, ".", ms=3, color="0.25"); m = max(th.max(), 16)
    ax.plot([0, m], [0, m], "k--", lw=1); ax.set_xlim(0, m); ax.set_ylim(0, 2 * m)
    ax.set_xlabel(r"$\chi^2_3$ quantiles"); ax.set_ylabel(r"$\|\mathcal{I}_n^{1/2}(\hat\lambda-\lambda_0)\|^2$")
    ax.set_title(f"p={p}, n={r['n'][i]:,}: joint Q–Q (log scale)", fontsize=10)
fig.tight_layout(); plt.show()

## 7. Consistency: estimates across replicates

Mean ± one standard deviation of $\hat\theta_j$ over the replicates, with the truth as a dashed line.

In [ ]:
fig, axes = plt.subplots(len(PS), 3, figsize=(12, 3 * len(PS)), squeeze=False)
for r, p in enumerate(PS):
    est, n, th0 = res[p]["est"], res[p]["n"], res[p]["theta0"]
    for j in range(3):
        ax = axes[r, j]; m, s = est[:, :, j].mean(1), est[:, :, j].std(1, ddof=1)
        ax.fill_between(n, m - s, m + s, color=sim.COL[j], alpha=0.2)
        ax.plot(n, m, "o-", color=sim.COL[j], ms=4); ax.axhline(th0[j], ls="--", color="k")
        ax.set_xscale("log"); ax.set_xlabel("$n$"); ax.set_title(f"p={p}: {sim.PAR_LABELS[j]}", fontsize=10)
fig.tight_layout(); plt.show()

## 8. Numbers quoted in the paper

Every number in Section 4 and Appendix D of the paper that comes from the simulations, in order of appearance. The Appendix D.9 checks are in Section 1, the local slopes of Appendix D.8 in Section 5, and the normality diagnostics of Appendix D.6 in Section 6.

### Table 1 and Sections 4.1–4.2

In [ ]:
sim.print_slope_table(res)

S = {p: sim.summarize(sim.to_log(res[p])) for p in PS}

print("\nEstimates on the boundary of Theta:")
total = 0
for p in PS:
    e, b = res[p]["est"], res[p]["bounds"]
    hit = np.isclose(e, b[:, 0], rtol=1e-3) | np.isclose(e, b[:, 1], rtol=1e-3)   # (n, replicate, parameter)
    total += hit.any(2).sum()
    for i, j in zip(*np.nonzero(hit.any(1))):
        print(f"   p={p}, n={res[p]['n'][i]}: {hit[i, :, j].sum()} estimate(s) of {['sigma^2', 'ell', 'tau^2'][j]}")
print(f"   {total} of {sum(res[p]['est'].shape[0] * res[p]['est'].shape[1] for p in PS)} fits have a coordinate on the boundary")

print("\nRMSE vs asymptotic SD of log sigma^2 and log ell, max over n of |RMSE / asy.SD - 1|:")
for p in PS:
    n = S[p]["n"]; dev = np.abs(S[p]["rmse"][:, :2] / S[p]["sd_fisher"][:, :2] - 1)
    print(f"   p={p}: all n {dev.max():.1%}, n >= 10^3 {dev[n >= 1000].max():.1%}")

print("\nFactor by which the RMSE falls from the smallest to the largest n (log scale):")
for p in PS:
    f = S[p]["rmse"][0] / S[p]["rmse"][-1]
    print(f"   p={p}: log sigma^2 {f[0]:.2f}, log ell {f[1]:.2f}, log tau^2 {f[2]:.0f}")

print("\nsqrt(n) RMSE(log tau^2) vs its limit sqrt(2), max relative deviation for n >= 3 x 10^3:")
for p in PS:
    n = S[p]["n"]; dev = np.abs(np.sqrt(n) * S[p]["rmse"][:, 2] / np.sqrt(2) - 1)
    print(f"   p={p}: {dev[n >= 3000].max():.1%}")
if 2 in PS:
    e = res[2]["est"][-1, :, 2] - res[2]["theta0"][2]
    print(f"   p=2, n=10^6: sqrt(n) RMSE(tau^2) on the original scale = {np.sqrt(res[2]['n'][-1] * np.mean(e ** 2)):.4f}"
          f" (limit sqrt(2) tau_0^2 = {np.sqrt(2) * res[2]['theta0'][2]:.5f})")

### Appendix D.3: profile likelihood of the extreme $p=1$ fits on 400 lengthscales

For the three largest $\hat\sigma^2$ at each $p=1$ sample size from $10^2$ to $3\times10^4$, the profile likelihood (maximised over $\sigma^2,\tau^2$) is evaluated on 400 log-spaced lengthscales. A nonnegative gap means that no grid point beats the computed MLE; the profile's local maxima on the grid are listed. This cell takes a few minutes.

In [ ]:
def profile_check(i, r):
    n = int(res[1]["n"][i])
    seed = sim.replicate_seeds(1, len(res[1]["n"]), res[1]["est"].shape[1])[i, r]
    Y = sim.simulate(1, n, sim.THETA0, np.random.default_rng(int(seed))); E = float(Y @ Y)
    ls = np.exp(np.linspace(*np.log(sim.BOUNDS["l"]), 400))
    prof = np.array([sim._profile(l, 1, Y, E, n)[0] for l in ls])
    s, l, t = res[1]["est"][i, r]
    f_mle = sim._nll_st(np.log(s), np.log(t), *sim._suffstats(1, Y, l), n, E)[0]
    pad = np.r_[np.inf, prof, np.inf]
    peaks = ls[(prof < pad[:-2]) & (prof < pad[2:])]
    return n, r, res[1]["est"][i, r], prof.min() - f_mle, peaks

if 1 in PS:
    cases = [(i, int(r)) for i in range(6) for r in np.argsort(res[1]["est"][i, :, 0])[-3:]]
    out = Parallel(n_jobs=-1)(delayed(profile_check)(i, r) for i, r in cases)
    for n, r, est, gap, peaks in out:
        print(f"n={n:>6d}, replicate {r:3d}: MLE (s, l, t) = {np.round(est, 3)}; "
              f"best grid point minus MLE = {gap:+.1e} (negative log-lik); profile maxima at l = {np.round(peaks, 3)}")
    print(f"\nMLE at least as good as all 400 grid points in {sum(g >= -1e-6 for _, _, _, g, _ in out)} of {len(out)} cases; "
          f"profile with more than one local maximum in {sum(len(pk) > 1 for *_, pk in out)} of {len(out)}")

### Appendix D.6–D.8

In [ ]:
if 3 in PS:
    z, _ = sim.standardized(sim.to_log(res[3]), len(res[3]["n"]) - 1)
    print("p=3, n=10^6, log scale: skewness (log s, log l) =", np.round(stats.skew(z[:, :2]), 2),
          "  KS distance =", np.round([stats.kstest(z[:, j], "norm").statistic for j in (0, 1)], 3))
if 1 in PS:
    r = S[1]["rmse"][:, :2] / S[1]["sd_fisher"][:, :2] - 1
    q = np.abs(S[1]["iqr"][:, :2] / S[1]["sd_fisher"][:, :2] - 1)
    print(f"p=1: RMSE / asy.SD - 1: log sigma^2 {r[0, 0]:.0%} at n={S[1]['n'][0]} -> {r[-1, 0]:.0%} at n={S[1]['n'][-1]};"
          f" log ell {r[0, 1]:.0%} -> {r[-1, 1]:.0%}")
    print(f"p=1: |IQR/1.349 / asy.SD - 1| between {q.min():.1%} and {q.max():.1%}")

# eigen-directions of R_n(l0) whose signal variance sigma_0^2 lambda_k exceeds tau_0^2
X = dense_design(3, 5); D2 = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)
lam = np.linalg.eigvalsh(np.exp(-D2 / (2 * sim.THETA0["l"] ** 2)))
print(f"p=3, n=125: {np.sum(sim.THETA0['s'] * lam > sim.THETA0['t'])} of 125 eigen-directions have signal variance above tau_0^2")
cnt = [int(np.sum(sim.THETA0["s"] * sim.spectrum(n, sim.THETA0["l"], np.zeros(n))[0] > sim.THETA0["t"])) for n in sim.N_LIST[1]]
print(f"p=1, n = 10^2 ... 10^6: {cnt} such directions")

print("asymptotic SDs at n = 10^6 (log scale):")
for p in PS:
    print(f"   p={p}: log sigma^2 {S[p]['sd_fisher'][-1, 0]:.3f}, log ell {S[p]['sd_fisher'][-1, 1]:.4f}")